# NDVI 可视化与统计分析

基于 Sentinel-2 SR HARMONIZED 数据，计算川西地区 NDVI 并进行交互式地图可视化和统计分析。

In [1]:
import ee
import geemap

from utils import cal_ndvi, cloud_mask_by_probability
from qiezi.ee_utils import image_float2int

## 1. 数据准备与 NDVI 计算

In [2]:
# 初始化
ee.Initialize(project='chaoqiezipython')

# 参数配置
region_dir = 'projects/chaoqiezipython/assets/region/Chuanxi'
s2_dir = 'COPERNICUS/S2_SR_HARMONIZED'
s2_cloud_dir = r'COPERNICUS/S2_CLOUD_PROBABILITY'
# region = ee.Geometry.Rectangle([103.4, 31.2, 103.8, 31.6])  # 川西某一区域
region = ee.FeatureCollection(region_dir)  # 川西地区
start_date = '2019-05-01'
end_date = '2019-09-30'
max_cloud_probability = 90  # 云概率阈值(%), 超过该概率视为云

D:\Softwares\anaconda3\envs\geo\Lib\site-packages\ee\deprecation.py:139: UserWarning: Unable to initialize deprecated assets: IncompleteRead(194676 bytes read, 6097 more expected)
  warnings.warn(f'Unable to initialize deprecated assets: {e}')


In [3]:
# 时间和空间过滤
filter_criteria = ee.Filter.And(
    ee.Filter.bounds(region),
    ee.Filter.date(start_date, end_date),
)
s2 = (ee.ImageCollection(s2_dir)
      .filter(filter_criteria)
      .select(['B4', 'B8']))
s2_cloud = ee.ImageCollection(s2_cloud_dir).filter(filter_criteria)

# 整合 s2 和 s2 cloud probability
match_criteria = ee.Filter.equals(leftField='system:index', rightField='system:index')
s2_with_cloud = ee.Join.saveFirst('cloud_mask').apply(
    primary=s2, secondary=s2_cloud, condition=match_criteria
)

# 像素级云掩膜
s2_masked = ee.ImageCollection(s2_with_cloud).map(cloud_mask_by_probability)

# 计算 NDVI
s2_ndvi = s2_masked.map(cal_ndvi)

# MVC 最大值合成和裁剪
ndvi_composite = s2_ndvi.select('NDVI').max().clip(region)

# scale缩放
ndvi_scaled = image_float2int(ndvi_composite, scale_factor=10000, int_type='int16')

print('NDVI处理完成.')

NDVI处理完成.


## 2. 交互式地图可视化

In [4]:
# NDVI 可视化参数
ndvi_vis = {
    # 'min': -0.2,
    # 'max': 0.9,
    'min': -10000,
    'max': 10000,
    'palette': [
        '#d73027',  # 极低NDVI (裸地/水体)
        '#fc8d59',  # 低NDVI
        '#fee08b',  # 中低NDVI
        '#d9ef8b',  # 中NDVI
        '#91cf60',  # 中高NDVI
        '#1a9850',  # 高NDVI (密集植被)
    ]
}

# 创建交互式地图
Map = geemap.Map()
Map.centerObject(region, zoom=6)
Map.addLayer(ndvi_scaled, ndvi_vis, 'NDVI (最大值合成)')
Map.addLayer(region, {'color': 'red'}, '研究区域', opacity=0.3)
Map.addLayerControl()
Map

Map(center=[30.72525493367449, 101.17628400154203], controls=(WidgetControl(options=['position', 'transparent_…

## 3. NDVI 统计分析

In [5]:
# 区域统计
stats = ndvi_composite.reduceRegion(
    reducer=ee.Reducer.mean()
    .combine(ee.Reducer.minMax(), sharedInputs=True)
    .combine(ee.Reducer.stdDev(), sharedInputs=True),
    geometry=region,
    scale=10,  # Sentinel-2 B4/B8 分辨率为 10m
    maxPixels=1e9
)

print(f'均值:   {stats.get("NDVI_mean").getInfo():.4f}')
print(f'最小值: {stats.get("NDVI_min").getInfo():.4f}')
print(f'最大值: {stats.get("NDVI_max").getInfo():.4f}')
print(f'标准差: {stats.get("NDVI_stdDev").getInfo():.4f}')

EEException: Image.reduceRegion: Too many pixels in the region. Found 3421860597, but maxPixels allows only 1000000000.
Ensure that you are not aggregating at a higher resolution than you intended; that is a frequent cause of this error. If not, then you may set the 'maxPixels' argument to a limit suitable for your computation; set 'bestEffort' to true to aggregate at whatever scale results in 'maxPixels' total pixels; or both.